# Sentiment Analysis - Aging Narratives Articles

This notebook applies **three sentiment analysis methods**:

1. **Lexicon-based** - Simple word counting (fast baseline)
2. **VADER** - Rule-based with ML enhancements (good accuracy, fast)
3. **DistilBERT** - Transformer-based deep learning (high accuracy)

---
## Step 1: Import Libraries

In [24]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# DistilBERT
from transformers import pipeline
import torch

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}")
print("Libraries imported successfully!")

Using device: CPU
Libraries imported successfully!


---
## Step 2: Load Data

In [25]:
INPUT_FILE = '../newData/cleanData/mergedData.csv'
OUTPUT_FILE = '../newData/cleanData/mergedData_sentiment_2.csv'

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} articles")

Loaded 34325 articles


---
## Step 3: Define All Lexicons and Functions

In [26]:
# ========================================
# LEXICONS
# ========================================
POSITIVE_WORDS = {
    'success', 'successful', 'thrive', 'thriving', 'achievement', 'accomplish',
    'wisdom', 'wise', 'experience', 'experienced', 'skilled', 'expert',
    'active', 'healthy', 'vibrant', 'energetic', 'vital', 'strong', 'strength',
    'independent', 'independence', 'capable', 'ability', 'able', 'competent',
    'happy', 'happiness', 'joy', 'joyful', 'positive', 'optimistic', 'hopeful',
    'contribute', 'contribution', 'productive', 'valuable', 'valued', 'respected',
    'opportunity', 'opportunities', 'growth', 'improve', 'improvement', 'benefit',
    'empower', 'empowering', 'empowered', 'resilient', 'resilience', 'innovative'
}

NEGATIVE_WORDS = {
    'crisis', 'burden', 'burdensome', 'strain', 'stress', 'pressure', 'overwhelming',
    'decline', 'declining', 'deteriorate', 'deteriorating', 'fail', 'failing', 'failure',
    'vulnerable', 'vulnerability', 'frail', 'frailty', 'fragile', 'weak', 'weakness',
    'problem', 'problems', 'challenge', 'challenging', 'difficult', 'struggle',
    'worry', 'worried', 'concern', 'concerned', 'fear', 'fearful', 'anxiety',
    'costly', 'expensive', 'shortage', 'lack', 'inadequate', 'insufficient',
    'risk', 'risky', 'danger', 'dangerous', 'threat', 'threatening',
    'tsunami', 'bomb', 'collapse', 'catastrophe', 'disaster', 'dire'
}

LIMITING_KEYWORDS = {'crisis', 'burden', 'tsunami', 'bomb', 'decline', 'problem', 'strain', 'collapse'}
EMPOWERING_KEYWORDS = {'active', 'successful', 'healthy', 'well', 'productive', 'wisdom', 'thrive', 'empower'}

# ========================================
# LEXICON SENTIMENT FUNCTION
# ========================================
def lexicon_sentiment(text):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, 'neutral'
    words = re.findall(r'\b\w+\b', str(text).lower())
    pos_count = sum(1 for w in words if w in POSITIVE_WORDS)
    neg_count = sum(1 for w in words if w in NEGATIVE_WORDS)
    total = pos_count + neg_count
    if total == 0:
        return 0.0, 'neutral'
    score = (pos_count - neg_count) / total
    if score > 0.15:
        label = 'positive'
    elif score < -0.15:
        label = 'negative'
    else:
        label = 'neutral'
    return round(score, 3), label

# ========================================
# VADER SENTIMENT FUNCTION
# ========================================
vader_analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, 'neutral', 0.0
    scores = vader_analyzer.polarity_scores(str(text))
    compound = scores['compound']
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    confidence = abs(compound)
    return round(compound, 3), label, round(confidence, 3)

print("Lexicons and functions defined!")
print(f"  Positive words: {len(POSITIVE_WORDS)}")
print(f"  Negative words: {len(NEGATIVE_WORDS)}")

Lexicons and functions defined!
  Positive words: 50
  Negative words: 52


---
## Step 4: Load DistilBERT Model

In [27]:
print("Loading DistilBERT model...")

distilbert_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
    truncation=True,
    max_length=512
)

def distilbert_sentiment(text):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, 'neutral', 0.0
    try:
        text_truncated = str(text)[:2000]
        result = distilbert_analyzer(text_truncated)[0]
        label = result['label'].lower()
        confidence = result['score']
        if label == 'positive':
            score = confidence
        else:
            score = -confidence
            label = 'negative'
        if confidence < 0.6:
            label = 'neutral'
        return round(score, 3), label, round(confidence, 3)
    except:
        return 0.0, 'neutral', 0.0

print("DistilBERT model loaded!")

Loading DistilBERT model...


c:\Users\Suha\Desktop\Suha\ML\longevityLab\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suha\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 266.20it/s, Materia

DistilBERT model loaded!


---
## Step 5: Apply Lexicon Sentiment Analysis

In [28]:
print("Applying Lexicon sentiment analysis...")

lexicon_results = df['article_text'].apply(lexicon_sentiment)
df['lexicon_score'] = lexicon_results.apply(lambda x: x[0])
df['lexicon_label'] = lexicon_results.apply(lambda x: x[1])

print("Done!")
print(df['lexicon_label'].value_counts())

Applying Lexicon sentiment analysis...
Done!
lexicon_label
negative    14334
positive    10347
neutral      9644
Name: count, dtype: int64


---
## Step 6: Apply VADER Sentiment Analysis

In [29]:
print("Applying VADER sentiment analysis...")

vader_results = df['article_text'].apply(vader_sentiment)
df['vader_score'] = vader_results.apply(lambda x: x[0])
df['vader_label'] = vader_results.apply(lambda x: x[1])
df['vader_confidence'] = vader_results.apply(lambda x: x[2])

print("Done!")
print(df['vader_label'].value_counts())

Applying VADER sentiment analysis...
Done!
vader_label
positive    24427
negative     9742
neutral       156
Name: count, dtype: int64


---
## Step 7: Apply DistilBERT Sentiment Analysis

**Note:** This takes 15-30 minutes on CPU.

In [30]:
print("Applying DistilBERT sentiment analysis...")
print("This may take 15-30 minutes on CPU")
print("-" * 50)

batch_size = 1000
total = len(df)
results = []

for i in range(0, total, batch_size):
    batch = df['article_text'].iloc[i:i+batch_size]
    batch_results = batch.apply(distilbert_sentiment)
    results.extend(batch_results.tolist())
    print(f"  Processed {min(i+batch_size, total)}/{total} ({min(i+batch_size, total)/total*100:.1f}%)")

distilbert_results = pd.Series(results)
df['distilbert_score'] = distilbert_results.apply(lambda x: x[0])
df['distilbert_label'] = distilbert_results.apply(lambda x: x[1])
df['distilbert_confidence'] = distilbert_results.apply(lambda x: x[2])

print("\nDone!")
print(df['distilbert_label'].value_counts())

Applying DistilBERT sentiment analysis...
This may take 15-30 minutes on CPU
--------------------------------------------------
  Processed 1000/34325 (2.9%)
  Processed 2000/34325 (5.8%)
  Processed 3000/34325 (8.7%)
  Processed 4000/34325 (11.7%)
  Processed 5000/34325 (14.6%)
  Processed 6000/34325 (17.5%)
  Processed 7000/34325 (20.4%)
  Processed 8000/34325 (23.3%)
  Processed 9000/34325 (26.2%)
  Processed 10000/34325 (29.1%)
  Processed 11000/34325 (32.0%)
  Processed 12000/34325 (35.0%)
  Processed 13000/34325 (37.9%)
  Processed 14000/34325 (40.8%)
  Processed 15000/34325 (43.7%)
  Processed 16000/34325 (46.6%)
  Processed 17000/34325 (49.5%)
  Processed 18000/34325 (52.4%)
  Processed 19000/34325 (55.4%)
  Processed 20000/34325 (58.3%)
  Processed 21000/34325 (61.2%)
  Processed 22000/34325 (64.1%)
  Processed 23000/34325 (67.0%)
  Processed 24000/34325 (69.9%)
  Processed 25000/34325 (72.8%)
  Processed 26000/34325 (75.7%)
  Processed 27000/34325 (78.7%)
  Processed 28000/34

---
## Step 8: Compare Methods

In [31]:
print("=" * 60)
print("SENTIMENT DISTRIBUTION COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Lexicon': df['lexicon_label'].value_counts(),
    'VADER': df['vader_label'].value_counts(),
    'DistilBERT': df['distilbert_label'].value_counts()
}).fillna(0).astype(int)

print(comparison)

print("\nCross-Method Agreement:")
all_agree = ((df['lexicon_label'] == df['vader_label']) & 
             (df['vader_label'] == df['distilbert_label'])).mean() * 100
print(f"  All three agree: {all_agree:.1f}%")

SENTIMENT DISTRIBUTION COMPARISON
          Lexicon  VADER  DistilBERT
negative    14334   9742       25046
neutral      9644    156         800
positive    10347  24427        8479

Cross-Method Agreement:
  All three agree: 29.6%


---
## Step 9: Create Ensemble & Predict Category

In [32]:
# Ensemble (majority voting)
def ensemble_sentiment(row):
    labels = [row['lexicon_label'], row['vader_label'], row['distilbert_label']]
    if labels.count('positive') >= 2:
        return 'positive'
    elif labels.count('negative') >= 2:
        return 'negative'
    elif labels.count('neutral') >= 2:
        return 'neutral'
    else:
        return row['vader_label']

df['ensemble_label'] = df.apply(ensemble_sentiment, axis=1)
df['ensemble_score'] = ((df['lexicon_score'] + df['vader_score'] + df['distilbert_score']) / 3).round(3)

# Predict category
def predict_category(row):
    text = str(row['article_text']).lower()
    words = set(re.findall(r'\b\w+\b', text))
    limiting_count = len(words & LIMITING_KEYWORDS)
    empowering_count = len(words & EMPOWERING_KEYWORDS)
    
    if row['ensemble_label'] == 'positive' and empowering_count > limiting_count:
        return 'empowering'
    elif row['ensemble_label'] == 'negative' and limiting_count > empowering_count:
        return 'limiting'
    elif row['ensemble_score'] > 0.2 and empowering_count >= 2:
        return 'empowering'
    elif row['ensemble_score'] < -0.2 and limiting_count >= 2:
        return 'limiting'
    else:
        return 'neutral'

df['predicted_category'] = df.apply(predict_category, axis=1)
df['category_match'] = df['phrase_category_clean'] == df['predicted_category']

print("Ensemble Distribution:")
print(df['ensemble_label'].value_counts())
print("\nPredicted Category Distribution:")
print(df['predicted_category'].value_counts())

Ensemble Distribution:
ensemble_label
positive    18534
negative    15581
neutral       210
Name: count, dtype: int64

Predicted Category Distribution:
predicted_category
neutral       20777
empowering     7505
limiting       6043
Name: count, dtype: int64


---
## Step 10: Evaluate Performance

In [33]:
print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

match_rate = df['category_match'].mean() * 100
print(f"\nOverall Match Rate: {match_rate:.1f}%")

# Per-method accuracy
def map_to_category(label):
    if label == 'positive': return 'empowering'
    elif label == 'negative': return 'limiting'
    else: return 'neutral'

lexicon_acc = (df['lexicon_label'].apply(map_to_category) == df['phrase_category_clean']).mean() * 100
vader_acc = (df['vader_label'].apply(map_to_category) == df['phrase_category_clean']).mean() * 100
distilbert_acc = (df['distilbert_label'].apply(map_to_category) == df['phrase_category_clean']).mean() * 100

print(f"\nPer-Method Accuracy:")
print(f"  Lexicon:     {lexicon_acc:.1f}%")
print(f"  VADER:       {vader_acc:.1f}%")
print(f"  DistilBERT:  {distilbert_acc:.1f}%")
print(f"  Ensemble:    {match_rate:.1f}%")

MODEL PERFORMANCE

Overall Match Rate: 60.2%

Per-Method Accuracy:
  Lexicon:     37.4%
  VADER:       23.3%
  DistilBERT:  21.3%
  Ensemble:    60.2%


---
## Step 11: Save Results

In [34]:
df['analysis_timestamp'] = datetime.now()
df.to_csv(OUTPUT_FILE, index=False)

print(f"Results saved to: {OUTPUT_FILE}")
print(f"Total records: {len(df)}")

Results saved to: ../newData/cleanData/mergedData_sentiment_2.csv
Total records: 34325


---
## Done!

New columns added:
- `lexicon_score`, `lexicon_label`
- `vader_score`, `vader_label`, `vader_confidence`
- `distilbert_score`, `distilbert_label`, `distilbert_confidence`
- `ensemble_score`, `ensemble_label`
- `predicted_category`, `category_match`